In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Let's see how embeddings work on a few examples.


In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

v1 is a vector, an array of 384 numbers. Each number stands for some concept the model learned. We can't read off what any one of them means. But two vectors with similar values point to texts about similar things.

Encode our document:

In [3]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

Next, we compare the query against the document using dot product:

In [4]:
v1.dot(dv)

np.float32(0.32332397)

We get 0.32.

Now we try an unrelated query:

In [5]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [6]:
v2.dot(dv)

np.float32(0.019730438)

And we get 0.01.

The first score for q1 vs d (0.32) is higher, so that query is more similar to the document about registration. The second score for q2 vs d sits near 0, because installing Docker has nothing to do with registration. A score near 0 means the two vectors are about as different as they can be.

That's the whole idea behind vector search: similar texts get similar vectors, and a dot product tells us how similar.

**Cosine similarity**

The all-MiniLM-L6-v2 model outputs normalized vectors - vectors with unit length. When both vectors are normalized, the dot product equals cosine similarity. That's why the model documentation says it "uses cosine similarity."

Cosine similarity measures the angle between two vectors, ignoring their length:

- 1.0 = same direction (similar)
- 0.0 = perpendicular (unrelated)
- -1.0 = opposite direction (opposite meaning)

Formally, if theta is the angle between two vectors, cosine similarity is cos(theta):

cos(0) = 1 - vectors point in the same direction
cos(90) = 0 - vectors are perpendicular
cos(180) = -1 - vectors point in opposite directions

Because our vectors are normalized, the dot product gives us cosine similarity directly. This is why we can use v1.dot(dv) to compare texts.

In practice, we rarely get cosine similarity below 0. The embedding model maps text to a region of the vector space where most vectors have positive components. There's no concept of "opposite meaning" that maps to a vector pointing the other way.


### Embedding Our Dataset

In the previous lesson we saw how embeddings work on a couple of examples. Now we apply them to the whole FAQ dataset.

**Loading the data**

In module 1 we created ingest.py for loading the FAQ data.

Download it into your project:
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

We use it here:

In [7]:
from ingest import load_faq_data

documents = load_faq_data()

**Generating embeddings**

Each document is a Python dictionary with a question and an answer. We embed both together. That way a query can match against the question text and the answer text in our index.

Build one text per document:

In [8]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

Now we generate the embeddings. We have about 1200 texts here. We won't hand the model all of them at once. That takes a long time, and we can't see what's happening inside. Instead we split them into batches.

First we import tqdm so we can watch the progress:

In [9]:
from tqdm.auto import tqdm

Next we chunk the dataset into batches of 50 and encode each batch:

In [10]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

We end up with 1350 vectors. On a GPU this is fast. Most of us run on Codespaces without a GPU, so it takes a bit, but it's a one-off.

We turn them into a 2-dimensional array (matrix) where

- rows are documents (vectors)
- columns are dimensions of the vectors

In [11]:
import numpy as np
X = np.array(vectors)

Calling X.shape returns (1208, 384) - number of documents vs number of dimensions.

### Vector Search

In the previous lesson we embedded our FAQ dataset into a matrix X with 1208 document vectors. Here we see how vector search works under the hood.

**Scoring documents**

We have a matrix X with all document embeddings. We take a query, compare it against every document, and keep the most similar ones.

When a query comes in, we embed it:

In [12]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

Next, we compute the dot product against all documents:

In [13]:
scores = X.dot(v_query)

This is matrix-vector multiplication. Each element i of scores is the cosine similarity between document i (row i of X) and v_query.

We could compute the same thing with a for loop:

In [14]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

But X.dot(v_query) is much faster. Numpy runs optimized C code instead of looping in Python, so the matrix version is hard to beat. The outcome is the same either way: one score per document.

### Best match

The highest score is the most similar document:


In [15]:
idx = np.argmax(scores)
idx, scores[idx]


(np.int64(2), np.float32(0.76294106))

This returns document 553 with score 0.76.

The index and score may differ for you. Our FAQ is a living document, so we add and remove entries over time.

Let's see which document it is:

In [16]:
documents[idx]
#We see:

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

**Top 5 results**


Usually we want more than the single best match, so let's pull the top 5.

np.argsort sorts from lowest to highest, so the last 5 are the top ones:

In [17]:
top5 = np.argsort(scores)[-5:]

They come out smallest-first, so we reverse them to get the highest first:

In [18]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

Now we can read off the top 5 scores:

In [33]:
for i in range (top5.size):
    print(documents[top5[i]])

{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has alr

There's a shorter trick I usually reach for. We negate the scores first, so the largest becomes the smallest. Then argsort puts the best matches at the front.
Here it is in one line:

In [34]:
#it is not working
top5 = np.argsort(-scores)[:5]

TypeError: bad operand type for unary -: 'list'


It looks cryptic the first time you see it. But it's a common way to turn a min-sort into a max-sort.

Let's read off the actual documents:

In [35]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294106
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions'


This is vector search in its simplest form. We embed the query, compute dot products against all documents, and return the highest-scoring ones.

We return 5 and not the single best for a reason. The answer to a question can be spread across several documents. One holds part of it, another fills in the rest. Sometimes the top result isn't the right one but the second is. We send all 5 to the LLM and let it combine them.

The number 5 is a starting point, picked on gut feeling. Later, when we evaluate search quality, we can test whether 3 or 10 works better for our data.

Doing this by hand with numpy is fine for a small dataset. A larger one needs a library that also handles filtering and ranking. That's what we turn to next.